# 3 · Check and submit

Notebook 2 chose a model on one split of the training images. Before submitting it, we
check four things, each of which could still sink it:

1. **Is it really as good as the leader?** On images that played no part in any choice.
2. **Does it cost what we think?** Counted by the scorer itself, not by our lab.
3. **Does it train in time?** On a 4-core CPU like GitHub's, the limit is 10 minutes.
4. **Where does it fail?** The mistakes it makes, next to the leader's.

Then we write `model.py` and submit.

## The choice

The cheapest setting that tied the leader in experiment 11, decided *before* any of the
checks below were run, and written down in `plans.py` as `FINAL`:

In [ ]:
import lab
import matplotlib.pyplot as plt
import pandas as pd
from plans import FINAL

final, leader = lab.leader(**FINAL), lab.leader()
rows = []
for name in ("widths", "optimizer", "lr", "weight_decay", "batch_size", "epochs", "low_res", "tta"):
    rows.append({"setting": name, "leader": getattr(leader, name), "ours": getattr(final, name)})
display(pd.DataFrame(rows).set_index("setting"))
for name, cfg in (("leader", leader), ("ours", final)):
    print(
        f"{name:6}: training {lab.train_flops(cfg, 21600) / 1e12:5.2f} TFLOP,"
        f" answering {lab.predict_flops(cfg) / 1e6:6.1f} MFLOP per image,"
        f" {lab.parameters(cfg):,} parameters"
    )

Everything else is the leader's: the same shape of network, the same turning and
mirroring, the same one-cycle schedule.

## 1 · Is it as good as the leader, on fresh images?

Every choice in notebook 2 was made on one split of the training images (`seed=1`).
A setting that won there is a little lucky on exactly those images: with dozens of
comparisons, some winners won by luck. So we check on two other splits (`seed=7`, the
one `uv run score` uses, and `seed=3`): new check images, and new training images too.
The leader and our choice, 3 runs each on each split, side by side. Nothing is chosen
from these runs. We also ran 45 and 75 epochs, to see the trade-off on fresh images.

In [ ]:
fresh = lab.load("12-confirm")
table = fresh.pivot_table(index="label", columns="split", values="accuracy", aggfunc="mean") * 100
table["both splits"] = fresh.groupby("label")["accuracy"].mean() * 100
table["train TFLOP"] = fresh.groupby("label")["train_flops_full"].mean() / 1e12
table["answer MFLOP"] = fresh.groupby("label")["predict_flops"].mean() / 1e6
order = ["leader", "45 epochs", "final: 60 epochs", "75 epochs"]
display(table.loc[order].round(2))
for label in order[1:]:
    print(lab.compare(fresh, "leader", label))

**Result.** On fresh images, our model scores 96.9% and the leader 97.1%: a difference
of 0.24 points, within luck (± 0.16, so 1.5 standard errors). On split 7 we are level
(97.2 vs 97.1); on split 3 we are half a point behind. **A tie, at a quarter of the
training compute and a seventh of the compute per image.**

Two things worth noticing:

- **Every model scores higher here than on the `seed=1` split** (the leader 97.1
  instead of 96.9). Some splits are simply easier. This is why we compare models on the
  *same* images, never a number from one split with a number from another.
- **The trade-off held on fresh images**: 45 epochs is clearly worse than the leader
  (−0.65, real), 75 epochs a little better (+0.16, luck). Had we wanted to *beat* the
  leader rather than match it, 75 epochs (a third of its compute) would be the choice.
  We decided on 60 before running these, and a decision made before seeing the result
  is the only one these runs can confirm.

## 2 · Did every part earn its place?

Each experiment in notebook 2 added one thing. An *ablation* looks back from the end:
take the finished model, undo one choice, keep everything else, and see what it was
worth. Where undoing a choice changes the cost per epoch, the epochs change so that
every row costs the same 6.5 TFLOP. We did the same for the leader, to see which of its
parts were worth their complication. All on the `seed=1` split, 3 runs each.

In [ ]:
def ablation(name, reference):
    t = lab.load(name)
    s = lab.summary(t)[["accuracy", "se", "train_tflop", "predict_mflop"]]
    s["training accuracy"] = (t.groupby("label", sort=False)["train_accuracy"].mean() * 100).round(
        2
    )
    s["vs " + reference] = [
        "" if label == reference else lab.compare(t, reference, label).split(": ")[1]
        for label in s.index
    ]
    return s


display(ablation("14-final-ablation", "final"))

Our model, one choice undone at a time:

- **The leader's width** (with the 16 epochs that fit): −2.0 points. The central
  decision of notebook 2: spend compute on epochs, not on width.
- **No turning and mirroring in training**: −1.1. Look at the training accuracy: 98.6%
  without them, against 96.1% with them. Without new orientations to learn from, a long
  training starts to memorize. (At 16 epochs, in experiment 2, it did not matter yet.)
- **All epochs at full size** (26 fit instead of 60): −0.7.
- **Batch 128**: −0.55, but noisy (± 0.34): probably real, not proven here.
- **AdamW instead of SGD**: −0.16, within luck. Experiment 9 found +0.57 for SGD at 45
  epochs; at 60 epochs, with the same 3 seeds, the difference is too small to see. We
  keep SGD (it is no worse), but the honest summary is that the optimizer matters less
  than it looked. One experiment is a hint, not a law: a gain found in one setting may
  shrink in another.

In [ ]:
display(ablation("13-leader-ablation", "leader"))
views = pd.DataFrame(list(lab.load("0-leader")["accuracy_by_views"])).astype(float).mean() * 100
print("the leader answering with 1, 2, 4, 8 orientations:", views.round(2).to_dict())

The leader, one part removed at a time:

- **Earned their place**: turning and mirroring in training (+1.3; without them it
  memorizes: 99.9% training accuracy), the max pool after the first layer (+1.0: moving
  4 pixels at a time instead skips pixels and loses texture), and the warm-up at the
  start of the one-cycle schedule (+0.65).
- **Did not**: the residual shortcuts (−0.08, luck: with only 3 blocks the network is
  shallow enough to train without them), weight decay (−0.18, luck), and orientations 5
  to 8 when answering (+0.02 for half of its compute per image).

Shortcuts are a good example of a complication copied from bigger networks, where they
are essential (a 50-layer network does not train without them), into a small one where
they are not. We kept them in our model: we did not test removing them there, and an
untested change is not a decision.

## The ladder: what each step bought

All on the `seed=1` split, each a mean of 3 to 6 runs, from the experiments of
notebook 2. Each rung keeps everything above it and changes one thing.

In [ ]:
def mean_of(experiment, label):
    t = lab.load(experiment)
    rows = t[t["label"] == label]
    return 100 * rows["accuracy"].mean(), rows["train_flops_full"].mean() / 1e12


ladder = pd.DataFrame(
    [
        ("the leader", *mean_of("0-leader", "leader")),
        ("half width, 16 epochs (exp 1)", *mean_of("1-size-and-time", "width 0.5 x 16 epochs")),
        (
            "+ batch 64, learning rate 0.024 (exp 5)",
            *mean_of("5-batch-and-lr", "batch 64, lr 0.024"),
        ),
        ("+ 3/4 of 45 epochs at 32x32 (exp 7)", *mean_of("8-regularize", "candidate")),
        ("+ SGD instead of AdamW (exp 9, 11)", *mean_of("11-sgd-budget", "45 epochs")),
        ("+ 60 epochs instead of 45 (exp 11)", *mean_of("11-sgd-budget", "60 epochs")),
    ],
    columns=["step", "accuracy", "train TFLOP"],
).set_index("step")
display(ladder.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ladder["train TFLOP"], ladder["accuracy"], "o-", color="tab:blue")
for i, (_, row) in enumerate(ladder.iterrows()):
    ax.annotate(
        f"{i}", (row["train TFLOP"], row["accuracy"]), textcoords="offset points", xytext=(6, -4)
    )
ax.axvline(26.22 / 5, color="gray", linestyle="--")
ax.text(26.22 / 5 * 1.03, 93.5, "1/5 of the leader", color="gray")
ax.set_xscale("log")
ax.set_xlabel("training compute (TFLOP, log scale)")
ax.set_ylabel("accuracy, seed=1 split (%)")
ax.set_title("From the leader (0) to our model (5)")
plt.show()

Halving the width and training for 16 epochs cut the compute by 6.6 and cost 3.5
points. Four changes that cost no compute (smaller batches, a higher learning rate,
small images first, SGD) won almost all of it back, and a few more epochs closed the
gap. Not one of them is a new idea; what made them work is that each was *measured*,
at equal compute, with enough runs to tell it from luck.

## 3 · Write the submission

GitHub only ever sees `model.py`, and it always uses its defaults. All our experiments
trained `bench.py`: the same code with every setting we tried, so that a notebook can
switch any of them on. A submission should not carry that weight. This cell writes
`model.py` with only what earned its place: 150 lines instead of 270, every one of
them used. Nothing in it is new; it is the bench with the unused settings taken out.

How do we know the lean copy is the model we tested? `tests/test_demo2.py` trains both
files from the same seed and checks that they end with **the same weights, to the last
bit**, give the same answers, and cost the same FLOPs. When you clean up code that you
have measured, write a test like that: it turns "I only tidied it" into a fact.

In [ ]:
%%writefile model.py
"""Demo 2: the leader's accuracy for a quarter of its compute.

A small residual network, trained with SGD for 60 epochs (the first 45 on half-size
images), that answers by averaging over 4 turns of each image. Every setting below was
chosen by an experiment: the notebooks in this folder show which one, and NOTES.md lists
them all. bench.py is the lab version of this file, with everything else we tried.

GitHub calls train(train_set, log), which uses the defaults in Config.
"""

import math
from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F


@dataclass(frozen=True)
class Config:
    widths: tuple[int, ...] = (12, 24, 48)  # channels per stage: half the leader's
    epochs: int = 60
    small_epochs: int = 45  # the first epochs see 32x32 images, at a quarter of the cost
    batch_size: int = 64
    lr: float = 0.1  # the peak of the one-cycle schedule
    warmup: float = 0.2  # share of training spent raising the learning rate to its peak
    momentum: float = 0.9
    weight_decay: float = 5e-4
    device: str = "cpu"  # "cuda" or "mps" to try settings faster; GitHub uses the CPU


class ResidualBlock(nn.Module):
    """Two 3x3 convolutions, and a shortcut that carries the input around them."""

    def __init__(self, c_in: int, c_out: int, stride: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:  # the input no longer fits: reshape it to match
            self.shortcut = nn.Sequential(
                nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


def make_network(widths: tuple[int, ...]) -> nn.Sequential:
    """A strided convolution and a max pool shrink 64x64 images to 16x16. Then one residual
    block per stage, each stage after the first at half the size. Then one score per class.
    """
    layers = [
        nn.Conv2d(3, widths[0], 3, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(widths[0]),
        nn.ReLU(),
        nn.MaxPool2d(2),
    ]
    c_in = widths[0]
    for stage, c_out in enumerate(widths):
        layers.append(ResidualBlock(c_in, c_out, stride=1 if stage == 0 else 2))
        c_in = c_out
    layers += [
        nn.AdaptiveAvgPool2d(1),  # average each channel over the whole image
        nn.Flatten(),
        nn.Linear(widths[-1], 10),
    ]
    return nn.Sequential(*layers)


# Turning and mirroring use flips and transposes only: they move numbers without any
# arithmetic, and the leaderboard's FLOP counter knows it.


def random_orientation(x: torch.Tensor) -> torch.Tensor:
    """Give each image in the batch one of its 8 orientations, at random."""

    def coin() -> torch.Tensor:  # True for about half of the images
        return torch.rand(len(x), 1, 1, 1, device=x.device) < 0.5

    x = torch.where(coin(), x.flip(3), x)  # mirror left-right
    x = torch.where(coin(), x.flip(2), x)  # mirror top-bottom
    x = torch.where(coin(), x.transpose(2, 3), x)  # swap rows and columns
    return x.contiguous()  # one tidy block of memory: convolutions run faster on it


def quarter_turns(x: torch.Tensor) -> list[torch.Tensor]:
    """The batch of images turned by 0, 90, 180 and 270 degrees."""
    quarter = x.transpose(2, 3).flip(2)
    return [x, quarter, x.flip(2, 3), quarter.flip(2, 3)]


class AverageOverTurns(nn.Module):
    """Score each image in all 4 quarter turns, and average the scores."""

    def __init__(self, network: nn.Module) -> None:
        super().__init__()
        self.network = network

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.stack([self.network(turned) for turned in quarter_turns(x)]).mean(0)


def train(train_set, log, cfg: Config = Config()) -> nn.Module:
    """Train on train_set (uint8 images, 0-255) and return a model that takes float images
    (0-1) and returns 10 class scores per image. log() draws the leaderboard's curves."""
    images = train_set.images.to(cfg.device)
    labels = train_set.labels.to(cfg.device)
    n = len(images)
    network = make_network(cfg.widths).to(cfg.device).train()
    optimizer = torch.optim.SGD(
        network.parameters(),
        cfg.lr,
        momentum=cfg.momentum,
        nesterov=True,
        weight_decay=cfg.weight_decay,
    )
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        cfg.lr,
        epochs=cfg.epochs,
        steps_per_epoch=math.ceil(n / cfg.batch_size),
        pct_start=cfg.warmup,
    )

    for epoch in range(cfg.epochs):
        # Running totals stay on the device: reading one (.item()) makes Python wait for it.
        total_loss = torch.zeros((), device=cfg.device)
        correct = torch.zeros((), device=cfg.device)
        for batch in torch.randperm(n, device=cfg.device).split(cfg.batch_size):
            x = random_orientation(images[batch].float() / 255)
            y = labels[batch]
            if epoch < cfg.small_epochs:
                x = F.avg_pool2d(x, 2)  # each 2x2 square of pixels becomes one
            scores = network(x)
            loss = F.cross_entropy(scores, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            schedule.step()
            total_loss += loss.detach() * len(batch)
            correct += (scores.argmax(1) == y).sum()
        log(train_loss=total_loss.item() / n, train_accuracy=correct.item() / n)

    return AverageOverTurns(network).cpu().eval()

## 4 · Check it the way GitHub will

`uv run score . --flops` trains `model.py` from scratch on this computer's CPU, exactly
as GitHub does (all 4 cores, no GPU), checks it on the `seed=7` split, and then trains
it a second time inside the scorer's own FLOP counter. It answers the three remaining
questions at once: is the compute what our lab counted, does it train in time, and
where does it fail. (About 10 minutes.)

In [ ]:
!uv run score . --flops

**What it says.**

- **Compute.** The scorer counts the training on the 17,280 images it trains on; on all
  21,600, as on GitHub, that is 1.25 times more: **6.5 TFLOP**, the number our lab
  predicted. **16 MFLOP per image.** Nothing went uncounted. (An earlier version turned
  images with `torch.rot90`, which the counter did not know and flagged as possibly
  heavy; it only moves numbers around, so we rewrote it with flips and a transpose,
  which the counter knows are free. A flag like that turns into a `*` on the
  leaderboard: worth avoiding in a submission meant to be compared.)
- **Time.** 206 seconds on this 4-core computer, well inside GitHub's 10 minutes. Most
  epochs are on small images, which also makes them faster.
- **Mistakes.** The same suspects as in the demo: herbaceous vegetation taken for
  permanent crop (fields with similar patterns), and highways and rivers taken for each
  other (long lines across the image). Forest, residential and sea are almost never
  wrong.

## Summary

| | the leader | ours | |
|---|---|---|---|
| accuracy, fresh splits (6 runs each) | 97.1% | 96.9% | a tie (−0.24 ± 0.16) |
| training compute | 26.2 TFLOP | 6.5 TFLOP | 4.0 times less |
| compute per image | 119 MFLOP | 16 MFLOP | 7.3 times less |
| parameters | 174,274 | 44,230 | 3.9 times fewer |

How we got there, in one sentence: **we counted the compute before spending it, and
then spent it on learning steps instead of width.** Half the width, smaller batches, a
higher learning rate, most epochs on small images, and SGD; each measured at equal
compute, with enough runs to tell a real gain from luck.

## Where to go from here

Ideas we did not try, roughly from most to least promising:

1. **The network still underfits** (training accuracy 96.1%, check accuracy 96.8%), and
   accuracy was still rising with epochs when we stopped (75 epochs: 97.3% on fresh
   images, at a third of the leader's compute). Anything that learns more per FLOP
   should help: for example three image sizes instead of two (16, 32, then 64 pixels),
   or a wider last stage, which is cheap because it works on 4x4 pixels.
2. **Drop the residual shortcuts**: they did not earn their place in the leader, and
   removing them saves a little compute. Test it on our model first.
3. **Answer from 2 orientations** instead of 4: half the compute per image (8 MFLOP),
   for about 0.2 points (experiment 0).
4. **Depthwise convolutions** (the MobileNet block, `block="inverted"` in `bench.py`)
   do far less arithmetic than full ones, but the FLOP counter charges their backward
   pass as if they were full convolutions, about 30 times what they compute. Under that
   counter they do not pay; if the counter changes, they become the first thing to try.
5. **Luck**: the same settings score up to 0.5 points apart with different seeds. One
   leaderboard score has about ± 0.3 points of luck in it: keep that in mind before
   celebrating (or mourning) a difference of a few tenths.

Every experiment here is a few lines in `plans.py`. Add yours, run it, and add a line to
`NOTES.md`.